In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
CHB02_DIR = (
    PROJECT_ROOT /
    "data" /
    "multi_patient"/
    "chb02"
)
SUMMARY_PATH = (
    CHB02_DIR /
    "chb02-summary.txt"
)
print("CHB02 Directory:")
print(CHB02_DIR)

print("\nSummary File:")
print(SUMMARY_PATH)

print(
    "\nSummary file exists:",
    SUMMARY_PATH.exists()
)


CHB02 Directory:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\multi_patient\chb02

Summary File:
C:\Users\Prajapati_Shivam\EEG-Seizure-Detection\data\multi_patient\chb02\chb02-summary.txt

Summary file exists: True


In [2]:
with open(
    SUMMARY_PATH,
    "r"
) as file:

    summary_text = file.read()

print(summary_text)


Data Sampling Rate: 256 Hz
*************************

Channels in EDF Files:
**********************
Channel 1: FP1-F7
Channel 2: F7-T7
Channel 3: T7-P7
Channel 4: P7-O1
Channel 5: FP1-F3
Channel 6: F3-C3
Channel 7: C3-P3
Channel 8: P3-O1
Channel 9: FP2-F4
Channel 10: F4-C4
Channel 11: C4-P4
Channel 12: P4-O2
Channel 13: FP2-F8
Channel 14: F8-T8
Channel 15: T8-P8
Channel 16: P8-O2
Channel 17: FZ-CZ
Channel 18: CZ-PZ
Channel 19: P7-T7
Channel 20: T7-FT9
Channel 21: FT9-FT10
Channel 22: FT10-T8
Channel 23: T8-P8

File Name: chb02_01.edf
File Start Time: 15:29:39
File End Time: 16:29:39
Number of Seizures in File: 0

File Name: chb02_02.edf
File Start Time: 17:29:49
File End Time: 18:29:49
Number of Seizures in File: 0

File Name: chb02_03.edf
File Start Time: 18:29:56
File End Time: 19:29:56
Number of Seizures in File: 0

File Name: chb02_04.edf
File Start Time: 19:30:03
File End Time: 20:30:03
Number of Seizures in File: 0

File Name: chb02_05.edf
File Start Time: 20:30:10
File End Time:

In [3]:
edf_files = sorted(
    CHB02_DIR.glob("*.edf")
)

print("Total EDF files:", len(edf_files))

print("\nAvailable EDF files:")

for file in edf_files:
    print(file.name)

Total EDF files: 8

Available EDF files:
chb02_01.edf
chb02_02.edf
chb02_03.edf
chb02_04.edf
chb02_05.edf
chb02_16+.edf
chb02_16.edf
chb02_19.edf


In [4]:
seizure_annotations = {
    "chb02_16.edf": [(130, 212)],
    "chb02_16+.edf": [(2972, 3053)],
    "chb02_19.edf": [(3369, 3378)]
}

print("Seizure-containing recordings:")

for file, intervals in seizure_annotations.items():
    print(f"{file}: {intervals}")

print("\nTotal seizure-containing recordings:",
      len(seizure_annotations))

Seizure-containing recordings:
chb02_16.edf: [(130, 212)]
chb02_16+.edf: [(2972, 3053)]
chb02_19.edf: [(3369, 3378)]

Total seizure-containing recordings: 3


In [5]:
import mne

print("MNE imported successfully!")

MNE imported successfully!


In [6]:
# ============================================================
# CHB02 PREPROCESSING TEST
# ============================================================

TEST_FILE = edf_files[0]

print("Testing file:", TEST_FILE.name)

raw = mne.io.read_raw_edf(
    TEST_FILE,
    preload=True,
    verbose=False
)

print("\nOriginal data shape:")
print(raw.get_data().shape)

print("\nSampling frequency:")
print(raw.info["sfreq"])

print("\nNumber of channels:")
print(len(raw.ch_names))

# Work on a copy
processed_raw = raw.copy()

# Band-pass filter
processed_raw.filter(
    l_freq=0.5,
    h_freq=40,
    verbose=False
)

print("\nPreprocessing completed successfully.")

print("\nProcessed data shape:")
print(processed_raw.get_data().shape)

Testing file: chb02_01.edf


C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\216225492.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(



Original data shape:
(23, 921600)

Sampling frequency:
256.0

Number of channels:
23

Preprocessing completed successfully.

Processed data shape:
(23, 921600)


In [7]:
# ============================================================
# VERIFY CHANNEL STRUCTURE ACROSS ALL CHB02 EDF FILES
# ============================================================

channel_summary = []

for file_path in edf_files:

    raw = mne.io.read_raw_edf(
        file_path,
        preload=False,
        verbose=False
    )

    channel_summary.append({
        "file": file_path.name,
        "n_channels": len(raw.ch_names),
        "sampling_frequency": raw.info["sfreq"],
        "channel_names": raw.ch_names
    })

print("CHB02 FILE CHANNEL VERIFICATION")
print("=" * 50)

for info in channel_summary:

    print("\nFile:", info["file"])
    print("Channels:", info["n_channels"])
    print(
        "Sampling Frequency:",
        info["sampling_frequency"]
    )

print("\n" + "=" * 50)

# Check whether channel count is consistent
channel_counts = set(
    info["n_channels"]
    for info in channel_summary
)

sampling_rates = set(
    info["sampling_frequency"]
    for info in channel_summary
)

print("Unique channel counts:", channel_counts)
print("Unique sampling rates:", sampling_rates)

if len(channel_counts) == 1 and len(sampling_rates) == 1:
    print("\nSUCCESS: All CHB02 files have compatible structure.")
else:
    print("\nWARNING: Some files have different channel structures.")

CHB02 FILE CHANNEL VERIFICATION

File: chb02_01.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_02.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_03.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_04.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_05.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_16+.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_16.edf
Channels: 23
Sampling Frequency: 256.0

File: chb02_19.edf
Channels: 23
Sampling Frequency: 256.0

Unique channel counts: {23}
Unique sampling rates: {256.0}

SUCCESS: All CHB02 files have compatible structure.


C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\543377313.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\543377313.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\543377313.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\543377313.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(
C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\543377313.py:9: RuntimeWarning: Cha

In [8]:
# ============================================================
# TEST COMPLETE PIPELINE ON ONE CHB02 RECORDING
# ============================================================

TEST_FILE = edf_files[0]

WINDOW_SIZE = 4  # seconds

frequency_bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 40)
}

feature_names = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

print("Testing complete pipeline on:")
print(TEST_FILE.name)

# Read EDF
raw = mne.io.read_raw_edf(
    TEST_FILE,
    preload=True,
    verbose=False
)

# Preprocess
raw.filter(
    l_freq=0.5,
    h_freq=40,
    verbose=False
)

# Get EEG data
data = raw.get_data()

sfreq = raw.info["sfreq"]

samples_per_window = int(
    WINDOW_SIZE * sfreq
)

print("\nData shape:", data.shape)
print("Sampling frequency:", sfreq)
print("Samples per window:", samples_per_window)

# Create first 4-second window
test_window = data[
    :,
    0:samples_per_window
]

print("Test window shape:", test_window.shape)

C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\3419788514.py:32: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


Testing complete pipeline on:
chb02_01.edf

Data shape: (23, 921600)
Sampling frequency: 256.0
Samples per window: 1024
Test window shape: (23, 1024)


In [9]:
# ============================================================
# RECREATE TEST WINDOW
# ============================================================

import mne

TEST_FILE = edf_files[0]

raw = mne.io.read_raw_edf(
    TEST_FILE,
    preload=True,
    verbose=False
)

raw.filter(
    l_freq=0.5,
    h_freq=40,
    verbose=False
)

data = raw.get_data()

sfreq = raw.info["sfreq"]

WINDOW_SIZE = 4

samples_per_window = int(
    WINDOW_SIZE * sfreq
)

test_window = data[
    :,
    0:samples_per_window
]

print("Test window recreated successfully!")
print("Test window shape:", test_window.shape)

C:\Users\Prajapati_Shivam\AppData\Local\Temp\ipykernel_20392\2387850678.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(


Test window recreated successfully!
Test window shape: (23, 1024)


In [10]:
import numpy as np
import pandas as pd
from scipy.signal import welch

# ============================================================
# FEATURE EXTRACTION FUNCTIONS
# ============================================================

def extract_statistical_features(window):

    channel_means = np.mean(window, axis=1)
    channel_stds = np.std(window, axis=1)
    channel_variances = np.var(window, axis=1)

    mean_feature = np.mean(channel_means)
    std_feature = np.mean(channel_stds)
    variance_feature = np.mean(channel_variances)

    return [
        mean_feature,
        std_feature,
        variance_feature
    ]


def extract_frequency_features(window, sfreq=256):

    channel_band_powers = []

    for channel in window:

        frequencies, psd = welch(
            channel,
            fs=sfreq,
            nperseg=512
        )

        band_powers = []

        for low_freq, high_freq in frequency_bands.values():

            mask = (
                (frequencies >= low_freq) &
                (frequencies < high_freq)
            )

            power = np.trapezoid(
                psd[mask],
                frequencies[mask]
            )

            band_powers.append(power)

        channel_band_powers.append(band_powers)

    # Average frequency-band power across channels
    return np.mean(
        channel_band_powers,
        axis=0
    )


# ============================================================
# EXTRACT FEATURES FROM TEST WINDOW
# ============================================================

stat_features = extract_statistical_features(
    test_window
)

freq_features = extract_frequency_features(
    test_window,
    sfreq
)

test_features = np.concatenate([
    stat_features,
    freq_features
])

print("FEATURE EXTRACTION SUCCESSFUL")
print("=" * 45)

for name, value in zip(
    feature_names,
    test_features
):
    print(f"{name}: {value}")

print("\nTotal features extracted:", len(test_features))

FEATURE EXTRACTION SUCCESSFUL
Mean: 3.623047154755146e-06
Std: 3.277740329332456e-05
Variance: 1.161025582409594e-09
Delta: 5.98195840419693e-10
Theta: 2.3951821579594966e-10
Alpha: 5.4931022766766093e-11
Beta: 1.946464555477906e-11
Gamma: 4.105105596445116e-12

Total features extracted: 8
